In [ ]:
# !pip install ijson
import os
import json
import glob
import time
import ijson
import gc
from typing import Any, Iterator, Dict, Set

In [ ]:
_INPUT_PRODUCTS_ = "./output/products/research_products_complete.json"
_OUTPUT_IDS_ = "./output/products/ids_research_products_orgs.txt"
CITATION_REL_TYPES = {"Cites", "IsCitedBy"}

In [ ]:
def iter_products(path):
    with open(path, "r", encoding="utf-8") as f:
        if path.lower().endswith(".ndjson"):
            for line in f:
                if line := line.strip():
                    try:
                        yield json.loads(line)
                    except json.JSONDecodeError:
                        continue
        else:
            yield from ijson.items(f, "item")

start = time.time()
uniq_ids = sorted({str(obj["id"]) for obj in iter_products(_INPUT_PRODUCTS_) if "id" in obj})
dt = time.time() - start

os.makedirs(os.path.dirname(_OUTPUT_IDS_), exist_ok=True)
with open(_OUTPUT_IDS_, "w", encoding="utf-8") as f:
    f.writelines(f"{uid}\n" for uid in uniq_ids)

print(f"Scanned {len(uniq_ids)} unique IDs in {dt:.1f}s and saved to: {_OUTPUT_IDS_}")

In [ ]:
def load_unique_ids(path: str) -> Set[str]:
    assert os.path.exists(path), f"File non trovato: {path}"
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}

def _is_ndjson(path: str) -> bool:
    if path.lower().endswith(".ndjson"):
        return True
    with open(path, "r", encoding="utf-8") as f:
        while True:
            c = f.read(1)
            if not c:
                return False
            if not c.isspace():
                return c != "["

def iter_json_objects(path: str) -> Iterator[Dict[str, Any]]:
    if _is_ndjson(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if isinstance(obj, dict):
                        yield obj
                except Exception:
                    continue
    else:
        with open(path, "r", encoding="utf-8") as f:
            for obj in ijson.items(f, "item"):
                if isinstance(obj, dict):
                    yield obj

def get_id_value(v: Any) -> str:
    if isinstance(v, str):
        return v.strip()
    if isinstance(v, dict):
        for k in ("id", "ID"):
            if k in v and isinstance(v[k], str):
                return v[k].strip()
    return ""

def extract_citations(unique_ids_file: str, rel_dir: str, out_file: str):
    ids = load_unique_ids(unique_ids_file)
    print(f"Loaded unique IDs: {len(ids)}")

    paths = sorted(glob.glob(os.path.join(rel_dir, "**", "*.json"), recursive=True))
    print(f"JSON files found: {len(paths)} in {rel_dir}")

    t0 = time.time()
    scanned = 0
    citations = set()
    
    for i, p in enumerate(paths, 1):
        print(f"[{i}/{len(paths)}] {p}", flush=True)
        for rel in iter_json_objects(p):
            scanned += 1
            rel_name = rel.get("relType", {}).get("name", "")
            if rel_name not in CITATION_REL_TYPES:
                continue
            
            src = get_id_value(rel.get("source"))
            tgt = get_id_value(rel.get("target"))
            
            if (src and src in ids) or (tgt and tgt in ids):
                citations.add((src, tgt) if rel_name == "Cites" else (tgt, src))

    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(list(citations), f, ensure_ascii=False)

    dt = time.time() - t0
    print(f"Scansionate: {scanned:,} | Citazioni uniche: {len(citations):,} | Tempo: {dt:.1f}s")
    print(f"Output: {out_file}")
    
def extract_partition(n: int):
    rel_dir = f"./relation_src/{n:02d}/relation"
    out_file = f"./output/citations/citations_{n:02d}.json"
    extract_citations(_OUTPUT_IDS_, rel_dir, out_file)
    gc.collect()

In [ ]:
for n in range(1, 15):
    extract_partition(n)

In [ ]:
# load and merge in one final file all partitions
partition_files = sorted(glob.glob("./output/citations/citations_*.json"))
partition = set()
for pf in partition_files:
    print("Loading partition file:", pf)
    with open(pf, "r", encoding="utf-8") as f:
        data = json.load(f)
        partition.update((tuple(item) for item in data))
with open("./output/citations/citations_merged.json", "w", encoding="utf-8") as f:
    json.dump([list(item) for item in partition], f, ensure_ascii=False)
print("Final citations saved in: ./output/citations/citations_merged.json")